# 01 · Acquire NASA OMNI2 hourly data
**Goal:** a verifiable raw snapshot: every annual file, its SHA-256, retrieval time, and the OMNIWeb documentation page that defines the Dst label tiers.

On Colab first run: `!git clone https://github.com/<YOU>/storm-time.git; %cd storm-time; !pip -q install pyarrow`

In [ ]:
import os, sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "stormtime").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "stormtime").exists(), "Run inside the cloned repo (see README)"
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stormtime import config as C, data as D, features as F, events as E, splits as S, baselines as B, metrics as M
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
RAW, CLEAN, FIG, RES = ROOT/"data/raw", ROOT/"data/clean", ROOT/"figures", ROOT/"results"
for p in (RAW, CLEAN, FIG, RES): p.mkdir(parents=True, exist_ok=True)
print("repo:", ROOT)

In [ ]:
YEARS = list(range(C.FIRST_YEAR, C.LAST_YEAR + 1))
doc = D.fetch_doc_snapshot(RAW)          # saves the OMNIWeb documentation page (provenance)
print(json.dumps(doc["availability"], indent=1))

**Check the printed availability table against the live page** (https://omniweb.gsfc.nasa.gov/html/ow_data.html). It sets `label_version` for every row, so it must not be hard-coded.

### Option B: if Colab cannot reach NASA (real files only)
Download `omni2_YYYY.dat` (or the single `omni2_all_years.dat`) from https://spdf.gsfc.nasa.gov/pub/data/omni/low_res_omni/ in your browser, upload to `data/raw/`, then run the cell below. Years already present are not downloaded again.

In [ ]:
# Option B only: uncomment what applies
# print(D.ingest_local_dat(RAW))                                   # omni2_YYYY.dat files placed in data/raw
# print(D.ingest_all_years_file(RAW / "omni2_all_years.dat", RAW, YEARS))   # one big file

In [ ]:
records = D.download_all(YEARS, RAW)
D.write_manifest(records, doc, RAW / "manifest.json")
man = pd.DataFrame(records)[["year", "n_lines", "bytes_raw", "sha256_raw", "http_last_modified", "retrieved_utc"]]
man.tail(8)

In [ ]:
raw = D.add_time(D.load_raw(RAW, YEARS))
val = D.validate_hourly(raw)
print(json.dumps(val, indent=1))
assert val["duplicates"] == 0 and val["monotonic"] and val["non_hourly_steps"] == 0, "time axis problem: paste this output back"
# loud parse sanity checks: a wrong column layout would break at least one of these
assert raw["hour"].between(0, 23).all() and raw["doy"].between(1, 366).all(), "year/doy/hour columns misparsed"
assert 300 < raw["speed"].median() < 600, f"speed median {raw['speed'].median()} looks wrong: column layout?"
assert -40 < raw["dst"].median() < 0, f"Dst median {raw['dst'].median()} looks wrong: column layout?"
assert abs(raw["bz_gsm"].median()) < 2, "Bz median should be near 0"
print("parse sanity checks passed")

### Coverage and fill decoding
Fraction of each key word that is missing after decoding per-word fill values, and where each stream actually ends.

In [ ]:
key = ["b_avg", "bz_gsm", "speed", "density", "pressure", "dst", "kp10", "ae"]
cov = pd.DataFrame({"missing_frac": raw[key].isna().mean().round(4),
                    "first_valid": [raw[k].first_valid_index() for k in key],
                    "last_valid": [raw[k].last_valid_index() for k in key]})
cov

In [ ]:
# sanity: value ranges after fill decoding (a leaked fill value would show up here)
raw[["speed", "density", "bz_gsm", "pressure", "dst"]].describe().T[["min", "50%", "max"]]

In [ ]:
json.dump({"validation": val, "coverage": cov.astype(str).to_dict()}, open(RES / "validation_01.json", "w"), indent=1)
print("saved", RES / "validation_01.json")

### Commit
```
git add data/raw results/validation_01.json && git commit -m "OMNI raw snapshot + manifest" && git push
```
Send me the printed `validation` JSON and the `cov` table if anything looks off (row counts per year, non-hourly steps, a leaked fill value).